In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import time
from google.colab import drive


drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/datasets/FlameVision.zip"

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall("dataset")


# otimizando as configurações
IMG_SIZE = (128, 128)  # reduzido para acelerar (era 224x224)
BATCH_SIZE = 128       # aumentado para processar mais rápido
EPOCHS = 10            # reduzido drasticamente (comecei com 50)

base_path = "dataset/FlameVision/archive/Classification"

TRAIN_PATH = os.path.join(base_path, 'train')
VAL_PATH = os.path.join(base_path, 'valid')
TEST_PATH = os.path.join(base_path, 'test')


def criar_modelo():
    """Modelo CNN"""
    model = models.Sequential([
        # camada 1 - entrada
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.1),

        # camada 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        # camada 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # camada 4
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),  # mais eficiente que Flatten

        # saída
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid')
    ])

    return model


# criando o modelo chamando a função
model = criar_modelo()

# compilar com otimizador rápido
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print(f"Tamanho da imagem: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Épocas: {EPOCHS}")
model.summary()


# data augmentation mais simples
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

valid_datagen = ImageDataGenerator(rescale=1./255)

# geradores regulares (mais rápido que o balanceado customizado)
train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=42
)

val_gen = valid_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_gen = valid_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f"classes: {train_gen.class_indices}")
print(f"amostras treino: {train_gen.samples}")
print(f"amostras validação: {val_gen.samples}")
print(f"amostras teste: {test_gen.samples}")


# otimização de callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,           # menos paciência para parar mais cedo
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,           # Reduz LR mais rápido
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'fast_model_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# medindo o tempo
start_time = time.time()

# treinando (sem parâmetro 'workers')
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"tempo de treinamento: {training_time:.1f} segundos")


# avaliação do conjunto de teste
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_gen, verbose=0)

print(f"   acurácia: {test_acc:.4f}")
print(f"   precisão: {test_precision:.4f}")
print(f"   recall: {test_recall:.4f}")

# calculando F1-Score
test_f1 = 2 * (test_precision * test_recall) / (test_precision + test_recall + 1e-7)
print(f"   F1-Score: {test_f1:.4f}")

# previsões
y_pred_probs = model.predict(test_gen, verbose=0)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = test_gen.classes

# relatório
print("\n resumo:")
print(classification_report(y_true, y_pred, target_names=['FIRE', 'NOFIRE'], digits=3))

# matriz de confusão
cm = confusion_matrix(y_true, y_pred)
print("matriz de confusão:")
print(f"      pred FIRE  pred NOFIRE")
print(f"FIRE     {cm[0,0]:5d}        {cm[0,1]:5d}")
print(f"NOFIRE   {cm[1,0]:5d}        {cm[1,1]:5d}")

# salvando modelo, caso a acurácia for boa
if test_acc > 0.85:
    model.save("fire_detection_fast_final.keras")
    print(f"\n modelo salvo. Acurácia: {test_acc:.2%}")
else:
    print(f"\n acurácia abaixo de 85% ({test_acc:.2%}).")

# resumo do treinamento
print("\n" + "="*60)
print("resumo do treinamento")
print("="*60)
print(f"tempo total: {training_time:.1f} segundos")
print(f"melhor acurácia validação: {max(history.history['val_accuracy']):.4f}")
print(f"acurácia final teste: {test_acc:.4f}")
print(f"épocas treinadas: {len(history.history['loss'])}")

# teste manual
print("\n testando com algumas imagens manualmente...")

def quick_predict(image_path, model):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    pred = model.predict(img_array, verbose=0)[0][0]
    return 'FIRE' if pred > 0.5 else 'NOFIRE', pred

# Testando com 2 imagens de cada classe
test_samples = []
for class_name in ['fire', 'nofire']:
    class_path = os.path.join(TEST_PATH, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if images:
        test_samples.append((os.path.join(class_path, images[0]), class_name))

for img_path, true_label in test_samples[:2]:
    pred_label, confidence = quick_predict(img_path, model)
    status = "ok" if pred_label.lower() == true_label else "erro"
    print(f"{status} {os.path.basename(img_path)}")
    print(f"   Verdadeiro: {true_label.upper()}")
    print(f"   Predito: {pred_label} (conf: {confidence:.2%})")

print("\n treinamento finalizado")

Mounted at /content/drive
Tamanho da imagem: (128, 128)
Batch size: 128
Épocas: 10


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 250,561 (978.75 KB)

 Trainable params: 249,857 (976.00 KB)

 Non-trainable params: 704 (2.75 KB)

Found 6800 images belonging to 2 classes.
Found 900 images belonging to 2 classes.
Found 900 images belonging to 2 classes.
classes: {'fire': 0, 'nofire': 1}
amostras treino: 6800
amostras validação: 900
amostras teste: 900


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8197 - loss: 0.4029 - precision: 0.8045 - recall: 0.8216
Epoch 1: val_accuracy improved from -inf to 0.99556, saving model to fast_model_best.keras
54/54 ━━━━━━━━━━━━━━━━━━━━ 485s 9s/step - accuracy: 0.8206 - loss: 0.4013 - precision: 0.8055 - recall: 0.8225 - val_accuracy: 0.9956 - val_loss: 0.5937 - val_precision: 0.9949 - val_recall: 0.9850 - learning_rate: 0.0010
Epoch 2/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9424 - loss: 0.1586 - precision: 0.9336 - recall: 0.9450
Epoch 2: val_accuracy did not improve from 0.99556
54/54 ━━━━━━━━━━━━━━━━━━━━ 477s 9s/step - accuracy: 0.9426 - loss: 0.1583 - precision: 0.9337 - recall: 0.9451 - val_accuracy: 0.2256 - val_loss: 0.8162 - val_precision: 0.2230 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9720 - loss: 0.0913 - precision: 0.9648 - recall: 0.9756
Epoch 3: val_accuracy did not improve from 0.99556
54/